In [ ]:
# Load the file(s) to run inference on.
# SimpleCNN1 expects the files to be greyscale spectrograms made from 5s .wav files
# It will convert the .wav files to 224px grayscale mel spectrograms, 
# then run inference using a pre-trained SimpleCNN1 model.

# FIRST RUN make_spectrograms_from_samples.ipynb to generate the spectrogram .png files.

## FILES TO PROCESS ##
folder_with_png_spectrograms = "Samples/"  # Folder containing .png spectrogram files

## MODEL TO LOAD ##
model_path = "SimpleCNN1_T1.pth"  # Path to the pre-trained model

In [10]:
# Load the model.

import os
import glob
from PIL import Image
import torch
from SimpleCNN1 import SimpleCNN

# Construct the model file and set its weights
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=2, img_height=224, img_width=224)
print("Model Architecture:")
print(model)
model.load_state_dict(torch.load(model_path, weights_only=True))
model.to(device)
model.eval()
print(f"Model loaded & set to eval() on device: {device}")


Model Architecture:
SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=100352, out_features=512, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=2, bias=True)
  )
)


RuntimeError: Attempting to deserialize object on CUDA device 1 but torch.cuda.device_count() is 1. Please use torch.load with map_location to map your storages to an existing device.

In [9]:
# Run inference on the images.
import torchvision.transforms as transforms

# Define preprocessing transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Get list of PNG files
png_files = glob.glob(os.path.join(folder_with_png_spectrograms, "*.png"))
print(f"Found {len(png_files)} PNG files")

# Class names (assuming binary classification: 0=not_smelly, 1=smelly)
# Claude Sonnet came up with this. Not me
class_names = ['smelly', 'not_smelly']

# Run inference on each image
for img_path in png_files:
    # Load and preprocess image
    image = Image.open(img_path).convert('RGB')  # Ensure RGB format cause SimpleCNN1 is silly and expects 3 channels
    input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
    
    # Run inference
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        predicted_class = torch.argmax(outputs, dim=1).item()
        confidence = probabilities[0][predicted_class].item()
    
    # Print results
    filename = os.path.basename(img_path)
    # Print table header once (when processing first file)
    if img_path == png_files[0]:
        print(f"{'Filename':<40} | {'Prediction':<12} | {'Confidence':<9}")
        print("-" * (40 + 3 + 12 + 3 + 9))
    # Print row
    print(f"{filename:<40} | {class_names[predicted_class]:<12} | {confidence:>9.3f}")

Found 6 PNG files
Filename                                 | Prediction   | Confidence
-------------------------------------------------------------------
MelodicaWithChocolate1.png               | not_smelly   |     1.000
PigSong1.png                             | not_smelly   |     1.000
test_961_human.png                       | not_smelly   |     1.000
test_926_AI.png                          | smelly       |     1.000
PigSong2.png                             | not_smelly   |     1.000
PigSong3.png                             | not_smelly   |     1.000
